# Analyse méthodologique du bassin du Vatancul

Ce notebook ne conserve que les étapes utiles à la **reproductibilité méthodologique** du cas d'étude :

1. contrôle du bassin et des données ;
2. préparation des MNT/MNS/MNH LiDAR HD ;
3. dérivés topographiques ;
4. calcul D8 et aire contributive ;
5. extraction des axes de concentration ;
6. détection de dépressions candidates ;
7. croisement avec les bâtiments et les routes ;
8. préparation éventuelle de la pluie observée.


> Les chemins de fichiers doivent être adaptés localement. Les données volumineuses ne sont pas destinées à être versionnées dans GitHub.


## 1. Configuration

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import fiona
import pyflwdir

from pyflwdir import dem
from rasterio.merge import merge
from rasterio.features import geometry_mask, shapes
from rasterio.transform import rowcol
from shapely.geometry import mapping, shape
from scipy import ndimage

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------------
# À ADAPTER
# ------------------------------------------------------------------
DATA_DIR = Path(r"D:\CHEMIN\VERS\LES_DONNEES")
OUTPUT_DIR = Path(r"D:\CHEMIN\VERS\LES_SORTIES")

BASSIN_GPKG = DATA_DIR / "BASSIN_VATENCUL_FINAL_BASE.gpkg"
BDTOPO_GPKG = DATA_DIR / "BDT_3-5_GPKG_LAMB93_D091-ED2026-06-15.gpkg"

# Dalles LiDAR HD IGN à 0,50 m
MNT_PATTERN = "LHD_*_MNT_O_0M50_*.tif"
MNS_PATTERN = "LHD_*_MNS_O_0M50_*.tif"
MNH_PATTERN = "LHD_*_MNH_O_0M50_*.tif"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RASTER_DIR = OUTPUT_DIR / "rasters"
VECTOR_DIR = OUTPUT_DIR / "vectors"
TABLE_DIR = OUTPUT_DIR / "tables"

for d in (RASTER_DIR, VECTOR_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

EPSG_TRAVAIL = 2154
NODATA = -9999.0
TAILLE_CELLULE_M = 0.50
SURFACE_CELLULE_M2 = TAILLE_CELLULE_M ** 2
SEUIL_AXES_M2 = 5000


## 2. Contrôle du bassin

In [ ]:
# Lecture de la couche bassin
layers = fiona.listlayers(BASSIN_GPKG)
print("Couches disponibles :", layers)

basin = gpd.read_file(BASSIN_GPKG, layer="bassin").to_crs(EPSG_TRAVAIL)
geom = basin.geometry.union_all()

print(f"Nombre d'entités : {len(basin)}")
print(f"CRS : {basin.crs}")
print(f"Géométrie valide : {bool(basin.geometry.is_valid.all())}")
print(f"Surface : {geom.area / 1e4:.3f} ha")
print("Emprise :", basin.total_bounds)


## 3. Mosaïque et découpe des modèles LiDAR

Les dalles MNT, MNS et MNH sont mosaïquées puis découpées sur le bassin avec un tampon de 100 m.  
La grille, la résolution et les altitudes d'origine sont conservées.


In [ ]:
def mosaic_clip(pattern: str, output_name: str, geometry, buffer_m=100):
    files = sorted(DATA_DIR.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Aucun fichier trouvé pour : {pattern}")

    clip_geom = geometry.buffer(buffer_m)
    srcs = [rasterio.open(p) for p in files]

    try:
        mosaic, transform = merge(
            srcs,
            bounds=clip_geom.bounds,
            nodata=NODATA
        )

        profile = srcs[0].profile.copy()
        profile.update(
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            transform=transform,
            count=1,
            dtype="float32",
            nodata=NODATA,
            compress="DEFLATE",
            tiled=True,
            blockxsize=512,
            blockysize=512,
            predictor=3,
            BIGTIFF="IF_SAFER",
        )

        arr = mosaic[0].astype("float32")

        inside = geometry_mask(
            [mapping(clip_geom)],
            out_shape=arr.shape,
            transform=transform,
            invert=True,
        )
        arr[~inside] = NODATA

        dst = RASTER_DIR / output_name
        with rasterio.open(dst, "w", **profile) as ds:
            ds.write(arr, 1)
            ds.update_tags(
                source="IGN LiDAR HD",
                processing="mosaic and polygon clip",
            )

        return dst, arr, profile

    finally:
        for src in srcs:
            src.close()


mnt_path, mnt, profile = mosaic_clip(
    MNT_PATTERN,
    "mnt_lidar_0p50m_bassin_plus100m.tif",
    geom,
)

mns_path, mns, _ = mosaic_clip(
    MNS_PATTERN,
    "mns_lidar_0p50m_bassin_plus100m.tif",
    geom,
)

mnh_path, mnh, _ = mosaic_clip(
    MNH_PATTERN,
    "mnh_lidar_0p50m_bassin_plus100m.tif",
    geom,
)

print("MNT :", mnt_path)
print("MNS :", mns_path)
print("MNH :", mnh_path)


## 4. Dérivés topographiques

In [ ]:
valid = mnt != NODATA
z = np.where(valid, mnt, np.nan)

# Remplissage local des NoData uniquement pour éviter les ruptures
# artificielles lors du calcul du gradient.
filled_local = np.where(
    valid,
    mnt,
    ndimage.generic_filter(
        np.where(valid, mnt, np.nan),
        np.nanmean,
        size=3,
        mode="nearest",
    ),
)

dzdy, dzdx = np.gradient(
    filled_local,
    TAILLE_CELLULE_M,
    TAILLE_CELLULE_M,
)

slope = np.degrees(
    np.arctan(np.hypot(dzdx, dzdy))
).astype("float32")
slope[~valid] = NODATA

aspect = (
    np.degrees(np.arctan2(-dzdx, dzdy)) + 360
) % 360
aspect = aspect.astype("float32")
aspect[~valid] = NODATA

azimuth = np.deg2rad(315)
altitude = np.deg2rad(45)
slope_rad = np.arctan(np.hypot(dzdx, dzdy))
aspect_rad = np.deg2rad(aspect)

hillshade = 255 * (
    np.sin(altitude) * np.cos(slope_rad)
    + np.cos(altitude)
    * np.sin(slope_rad)
    * np.cos(azimuth - aspect_rad)
)
hillshade = hillshade.clip(0, 255).astype("float32")
hillshade[~valid] = NODATA


def write_raster(name, array, unit):
    prof = profile.copy()
    prof.update(
        dtype="float32",
        nodata=NODATA,
        compress="DEFLATE",
        predictor=3,
    )
    path = RASTER_DIR / f"{name}.tif"
    with rasterio.open(path, "w", **prof) as ds:
        ds.write(array, 1)
        ds.update_tags(unit=unit, source="MNT LiDAR HD IGN")
    return path


write_raster("pente_degres", slope, "degree")
write_raster("aspect_degres", aspect, "degree")
write_raster("ombrage_315_45", hillshade, "0-255")


## 5. Direction D8 et aire contributive

Le MNT est rempli pour obtenir une surface continue utilisée uniquement pour le diagnostic topographique.  
La direction d'écoulement D8 et l'aire contributive sont ensuite calculées. Ces résultats décrivent une **concentration topographique potentielle** et ne constituent pas une simulation hydraulique.


In [ ]:
filled_dem, d8 = dem.fill_depressions(
    mnt.astype("float32"),
    outlets="edge",
    nodata=NODATA,
    max_depth=-1.0,
    connectivity=8,
)

flw = pyflwdir.from_array(
    d8,
    ftype="d8",
    transform=profile["transform"],
    latlon=False,
    check_ftype=False,
)

uparea = flw.upstream_area(unit="m2").astype("float32")
uparea[~valid] = NODATA

filldepth = np.where(
    valid,
    filled_dem - mnt,
    NODATA,
).astype("float32")

write_raster(
    "mnt_rempli_diagnostic",
    filled_dem.astype("float32"),
    "m",
)
write_raster(
    "aire_contributive_d8_m2",
    uparea,
    "m2",
)
write_raster(
    "profondeur_remplissage_depressions_m",
    filldepth,
    "m",
)


## 6. Axes de concentration topographique

In [ ]:
inside_basin = (
    geometry_mask(
        [mapping(geom)],
        out_shape=mnt.shape,
        transform=profile["transform"],
        invert=True,
    )
    & valid
)

stream_mask = (
    (uparea >= SEUIL_AXES_M2)
    & inside_basin
)

features = flw.streams(mask=stream_mask)
axes = gpd.GeoDataFrame.from_features(
    features,
    crs=f"EPSG:{EPSG_TRAVAIL}",
)

if len(axes):
    axes = axes[axes.geometry.notna()].copy()
    axes["geometry"] = axes.geometry.intersection(geom)
    axes = (
        axes[~axes.geometry.is_empty]
        .explode(index_parts=False)
        .reset_index(drop=True)
    )

    axes["id_axe"] = np.arange(1, len(axes) + 1)
    axes["longueur_m"] = axes.length
    axes["seuil_m2"] = SEUIL_AXES_M2
    axes["interpretation"] = (
        "axe de concentration topographique potentiel"
    )

    axes.to_file(
        VECTOR_DIR / "axes_concentration_d8.gpkg",
        layer="axes_d8",
        driver="GPKG",
    )

    print(
        f"{len(axes)} tronçons ; "
        f"{axes.length.sum()/1000:.3f} km"
    )
else:
    print("Aucun axe extrait.")


## 7. Dépressions candidates

In [ ]:
# Critères utilisés pour une présélection :
# profondeur >= 0,10 m et aire >= 10 m².
mask_dep = (
    (filldepth >= 0.10)
    & inside_basin
)

labels, _ = ndimage.label(
    mask_dep,
    structure=np.ones((3, 3), dtype=int),
)

counts = np.bincount(labels.ravel())
minimum_cells = int(np.ceil(10 / SURFACE_CELLULE_M2))

keep_ids = np.where(counts >= minimum_cells)[0]
keep_ids = keep_ids[keep_ids != 0]
keep = np.isin(labels, keep_ids)

records = []

for geom_json, value in shapes(
    labels.astype("int32"),
    mask=keep,
    transform=profile["transform"],
    connectivity=8,
):
    label_id = int(value)
    if label_id == 0 or counts[label_id] < minimum_cells:
        continue

    pix = labels == label_id
    depths = filldepth[pix]

    records.append({
        "id_dep": label_id,
        "area_m2": counts[label_id] * SURFACE_CELLULE_M2,
        "volume_fill_m3": float(
            depths.sum() * SURFACE_CELLULE_M2
        ),
        "depth_max_m": float(depths.max()),
        "depth_med_m": float(np.median(depths)),
        "interpretation": "dépression topographique candidate",
        "geometry": shape(geom_json),
    })

depressions = gpd.GeoDataFrame(
    records,
    crs=f"EPSG:{EPSG_TRAVAIL}",
)

if len(depressions):
    depressions = depressions.sort_values(
        "volume_fill_m3",
        ascending=False,
    ).reset_index(drop=True)

    depressions.to_file(
        VECTOR_DIR / "depressions_candidates.gpkg",
        layer="depressions",
        driver="GPKG",
    )

print("Dépressions candidates :", len(depressions))


## 8. Croisement avec les enjeux territoriaux

Les bâtiments et les routes sont utilisés pour une **présélection** des secteurs à inspecter.  
La proximité d'un bâtiment ou le croisement d'une route avec un axe D8 ne prouve pas une inondation.


In [ ]:
def read_clip(layer_name):
    g = gpd.read_file(
        BDTOPO_GPKG,
        layer=layer_name,
        bbox=geom.bounds,
        engine="pyogrio",
    )
    g = g[g.intersects(geom)].copy()
    g["geometry"] = g.geometry.intersection(geom)
    return g[~g.geometry.is_empty]


batiments = read_clip("batiment")
routes = read_clip("troncon_de_route")

if len(axes):
    axes_union = axes.geometry.union_all()

    batiments["dist_axe_m"] = (
        batiments.geometry.distance(axes_union)
    )
    batiments["proche_axe_10m"] = (
        batiments["dist_axe_m"] <= 10
    )

    routes["croise_axe_d8"] = (
        routes.intersects(axes_union)
    )
else:
    batiments["dist_axe_m"] = np.nan
    batiments["proche_axe_10m"] = False
    routes["croise_axe_d8"] = False

batiments.to_file(
    VECTOR_DIR / "enjeux_d8.gpkg",
    layer="batiments",
    driver="GPKG",
)

routes.to_file(
    VECTOR_DIR / "enjeux_d8.gpkg",
    layer="routes",
    driver="GPKG",
)

print("Bâtiments :", len(batiments))
print(
    "Bâtiments à 10 m ou moins d'un axe :",
    int(batiments["proche_axe_10m"].sum()),
)
print("Routes :", len(routes))
print(
    "Tronçons routiers croisant un axe :",
    int(routes["croise_axe_d8"].sum()),
)


## 9. Pluie observée — traitement optionnel

Cette partie permet de préparer une série pluviométrique à pas de 6 minutes à partir des fichiers Météo-France.  
Elle reste indépendante du calcul D8. Elle sert à documenter l'événement utilisé dans la simulation hydraulique.


In [ ]:
# À activer si les fichiers Météo-France sont disponibles.
METEO_PATTERN = "MN_91_*.csv.gz"
STATION = 91275001

meteo_files = sorted(DATA_DIR.glob(METEO_PATTERN))

if meteo_files:
    parts = []

    for path in meteo_files:
        df = pd.read_csv(
            path,
            sep=";",
            compression="gzip",
            usecols=[
                "NUM_POSTE",
                "NOM_USUEL",
                "AAAAMMJJHHMN",
                "RR",
                "QRR",
            ],
            low_memory=False,
        )

        df = df[df["NUM_POSTE"] == STATION].copy()
        if len(df):
            parts.append(df)

    if parts:
        rain = (
            pd.concat(parts, ignore_index=True)
            .drop_duplicates(["AAAAMMJJHHMN"])
            .sort_values("AAAAMMJJHHMN")
        )

        rain["datetime_utc"] = pd.to_datetime(
            rain["AAAAMMJJHHMN"].astype(str),
            format="%Y%m%d%H%M",
            errors="coerce",
        )

        rain = rain.set_index("datetime_utc").sort_index()

        # Exemple : événement des 9–10 octobre 2024.
        event = rain.loc[
            "2024-10-09":"2024-10-10",
            ["NUM_POSTE", "NOM_USUEL", "RR", "QRR"],
        ].copy()

        event.to_csv(
            TABLE_DIR / "pluie_gometz_09_10_octobre_2024.csv"
        )

        print(
            "Cumul sur la fenêtre extraite :",
            float(event["RR"].sum()),
            "mm",
        )
else:
    print("Aucun fichier météo trouvé : étape ignorée.")


## 10. Tableau de synthèse

In [ ]:
metrics = {
    "surface_bassin_ha": geom.area / 1e4,
    "seuil_axes_m2": SEUIL_AXES_M2,
    "axes_longueur_km": (
        axes.length.sum() / 1000 if len(axes) else 0
    ),
    "depressions_candidates_n": len(depressions),
    "batiments_n": len(batiments),
    "batiments_proches_axes_10m_n": int(
        batiments["proche_axe_10m"].sum()
    ),
    "routes_n": len(routes),
    "routes_croisant_axes_n": int(
        routes["croise_axe_d8"].sum()
    ),
}

pd.DataFrame([metrics]).to_csv(
    TABLE_DIR / "indicateurs_methodologiques_vatencul.csv",
    index=False,
)

print(json.dumps(metrics, indent=2, ensure_ascii=False))


## Lecture des résultats

Les sorties de ce notebook doivent être interprétées comme des **indicateurs topographiques de présélection** :

- les axes D8 représentent des cheminements potentiels liés au relief ;
- les dépressions sont des formes topographiques candidates ;
- les bâtiments proches et les routes croisées sont des objets à contrôler ;
- ces résultats ne remplacent pas une modélisation hydraulique calibrée ni une observation de terrain.

La simulation HEC-RAS, la publication GeoServer/MapStore et la rédaction du mémoire sont volontairement exclues de ce notebook afin de conserver un document court, lisible et centré sur la méthode.
